# Build a living tree spec

Run the setup, then work in the embedded view.

In [ ]:
from pathlib import Path
import sys

# Use this checkout from demos/, spytial-py/, or spytial-org/.
cwd = Path.cwd()
repo = next((p for p in (cwd, cwd.parent, cwd / "spytial-py")
             if (p / "spytial" / "suggest" / "_session.py").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook from the spytial-py checkout or its demos folder.")
sys.path.insert(0, str(repo))

from dataclasses import dataclass
from typing import Optional
from hypothesis import strategies as st
from IPython.display import display
import spytial
from spytial.suggest import suggest, SuggestSession, SpecUnavailableError

In [ ]:
@dataclass
class Tree:
    value: int
    left: Optional["Tree"] = None
    right: Optional["Tree"] = None

examples = [
    Tree(8, Tree(3), Tree(10)),
    Tree(8, Tree(3, Tree(1), Tree(6)), Tree(10, None, Tree(14))),
]

# Generate small trees without cycles or shared children.
def trees(depth):
    children = st.none() if depth == 0 else st.one_of(st.none(), trees(depth - 1))
    return st.builds(Tree, value=st.integers(0, 20), left=children, right=children)

strategy = trees(4)

## Open the session

Enter **Parents above children**, then click **Explore**. Drag and rate both diagrams.

In [ ]:
session = suggest(Tree, examples=examples, strategy=strategy, max_examples=10)
spec = session.spec
display(session)

## Finish a round

Click **I'm done** in the view. A ✓ marks an accepted requirement. Without enough evidence, the round stays unfinished; use **Resume** to continue.

In [ ]:
[(r.prompt, r.status) for r in session.requirements]

## Apply it

This also shows what happens if you finish before accepting any requirement.

In [ ]:
try:
    spec = session.apply()
except SpecUnavailableError as error:
    print(error)
else:
    spytial.diagram(examples[1])

## Read the spec

Each rule names its prompt and accepted interpretation.

In [ ]:
print(spec)

In [ ]:
spec.provenance

## Add another requirement

In the **same view**, enter **Align siblings horizontally**. The accepted rules still apply while you drag and vote. Click **I'm done**, then run this again:

In [ ]:
spytial.diagram(examples[1])
print(spec)

## Revisit a requirement

Click its prompt above the diagrams. Revise the wording or choose **Continue**. Earlier classifications remain in **History**, where you can correct them.

In [ ]:
[(r.prompt, r.status, len(r.history)) for r in session.requirements]

## Inspect generation time

Seconds spent on the latest request, with subsequent comparison searches accumulated.

In [ ]:
session.timings

## Save progress

Saving keeps the controls open.

In [ ]:
save_path = Path("tree-session.json")
session.save(save_path)

## Close the controls

Run this when finished. The accepted spec remains usable.

In [ ]:
session.close()
spytial.diagram(examples[1], spec=spec)

## Reopen

Continue in the new view. Apply the restored spec explicitly when ready.

In [ ]:
session = SuggestSession.load(Tree, save_path, strategy=strategy, max_examples=10)
spec = session.spec
display(session)

For scripts, the same controls are available as `session.add_requirement(text)`, `session.done()`, `requirement.revise(text)`, and `requirement.resume()`. To change search bounds, use `search=HypothesisSearch(strategy, max_seconds=5)` with `max_examples=10` instead of `strategy=`.